In [1]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

In [2]:
# Database connection details
db_username = "postgres"
db_password = "DevPost"
db_host = "192.168.0.166"
db_port = "5432"
db_name = "meal_price_trends"
table_name = "meal_price_merged_table"

# Create the connection URL
connection_url = f"postgresql+psycopg2://{db_username}:{db_password}@{db_host}:{db_port}/{db_name}"

# Create the SQLAlchemy engine
engine = create_engine(connection_url)

# Query to fetch data
query = f"SELECT * FROM {table_name}"

# Fetch data into a DataFrame
try:
    mergedDf = pd.read_sql(query, con=engine)
    print("Data successfully loaded into the DataFrame.")
except Exception as e:
    print("Error:", e)
finally:
    engine.dispose()

Data successfully loaded into the DataFrame.


In [3]:
mergedDf.head()

,state,avg_delinquent_debt,avg_student_loan_debt,avg_housing_cost_burden,avg_emergency_savings,avg_credit_score,avg_foreclosure,avg_health_insurance,avg_net_worth,avg_homeownership_rate,...,percent_gap_cost_snap,cost_exceeds_tfp,gap_cost_snap_plus_15,percent_gap_cost_snap_plus_15,cost_exceeds_tfp_plus_15,cost_exceeds_tfp_plus_25,normalized_cost_per_meal,cost_per_capita,total_gap_cost,region
0,AK,0.25228,0.094395,0.691931,0.679253,0.660615,0.000929,0.892573,0.110953,0.657373,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None
1,AL,0.32122,0.083597,0.538261,0.623578,0.551521,0.000683,0.908868,0.078902,0.721031,...,0.10,1.0,-0.10,-0.05,0.0,0.0,0.130653,57060.72,5019.23,South
2,AL,0.32122,0.083597,0.538261,0.623578,0.551521,0.000683,0.908868,0.078902,0.721031,...,0.06,1.0,-0.17,-0.07,0.0,0.0,0.115578,49946.40,3091.92,South
3,AL,0.32122,0.083597,0.538261,0.623578,0.551521,0.000683,0.908868,0.078902,0.721031,...,0.07,1.0,-0.16,-0.07,0.0,0.0,0.118928,201806.73,13390.02,South
4,AL,0.32122,0.083597,0.538261,0.623578,0.551521,0.000683,0.908868,0.078902,0.721031,...,0.21,1.0,0.13,0.06,1.0,0.0,0.180905,40947.87,7195.86,South


In [4]:
import dash
from dash import dcc, html, Input, Output, ctx
import dash_bootstrap_components as dbc
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# Load the dataset
data = mergedDf

# Initialize Dash app with Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.LUX], suppress_callback_exceptions=True)
app.title = "Food Insecurity Dashboard"

# Layout of the app with Navbar
app.layout = dbc.Container([
    # Header Title
    html.H1("Food Insecurity and Socioeconomic Disparities Dashboard", className="text-center my-4"),
    
    # Navbar
    dbc.Navbar(
        dbc.Container([
            dbc.Nav(
                [
                    dbc.NavLink("Income Levels vs Food Insecurity", href="/tab-1", id="tab-1-link", active="exact"),
                    dbc.NavLink("Racial Disparities", href="/tab-2", id="tab-2-link", active="exact"),
                    dbc.NavLink("Regional Insights", href="/tab-3", id="tab-3-link", active="exact"),
                    dbc.NavLink("Homeownership Disparities", href="/tab-4", id="tab-4-link", active="exact"),
                    dbc.NavLink("SNAP Coverage & Financial Indicators", href="/tab-5", id="tab-5-link", active="exact"),
                    dbc.NavLink("Savings & Food Insecurity", href="/tab-6", id="tab-6-link", active="exact"),
                ],
                pills=True
            )
        ]),
        color="primary",
        dark=True
    ),

    # Content container
    html.Div(id="page-content", className="p-4"),
], fluid=True)

# Callback to handle navigation and dynamic content rendering
@app.callback(
    Output("page-content", "children"),
    [Input("tab-1-link", "n_clicks"),
     Input("tab-2-link", "n_clicks"),
     Input("tab-3-link", "n_clicks"),
     Input("tab-4-link", "n_clicks"),
     Input("tab-5-link", "n_clicks"),
     Input("tab-6-link", "n_clicks")]
)
def display_page(n1, n2, n3, n4, n5, n6):
    triggered_id = ctx.triggered_id

    # Income Levels vs Food Insecurity
    if triggered_id == "tab-1-link" or not triggered_id:
        fig = px.scatter(
            data, x="avg_housing_cost_burden", y="percent_gap_cost_snap",
            size="avg_emergency_savings", color="region",
            title="Income Levels vs Food Insecurity (Bubble Chart)"
        )
        return dcc.Graph(figure=fig)

    # Racial Disparities
    elif triggered_id == "tab-2-link":
        racial_columns = ['avg_aapi', 'avg_black', 'avg_hispanic', 'avg_white']
        grouped_data = data.groupby('state')[racial_columns].mean().reset_index()
        melted_data = grouped_data.melt(id_vars=['state'], var_name='Race', value_name='Percentage')
        fig = px.choropleth(
            melted_data,
            locations="state",
            locationmode="USA-states",
            color="Percentage",
            scope="usa",
            animation_frame="Race",
            title="Racial Disparities Across States",
            height=600,
            width=1000
        )
        fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0}, title_x=0.5)
        return dcc.Graph(figure=fig, style={"height": "700px"})

    # Regional Insights
    elif triggered_id == "tab-3-link":
        dropdown_control = html.Div([
            html.Label("Select Graph Type:", className="form-label"),
            dcc.Dropdown(
                id="regional-chart-toggle",
                options=[
                    {"label": "Heatmap of Food Costs", "value": "heatmap"},
                    {"label": "Box Plot of Regional Variability", "value": "boxplot"}
                ],
                value="heatmap",
                clearable=False
            )
        ])
        graph_output = html.Div(id="regional-insights-graph")
        return html.Div([
            html.H5("Regional Insights", className="mb-4"),
            dropdown_control,
            graph_output
        ])

    # Homeownership Disparities
    elif triggered_id == "tab-4-link":
        toggle_control = html.Div([
            dbc.Switch(
                id="chart-toggle",
                label="Enable 3D Surface Chart",
                value=False,
                className="mb-3"
            )
        ])
        graph_output = html.Div(id="homeownership-graph")
        return html.Div([
            html.H5("Homeownership Disparities", className="mb-4"),
            toggle_control,
            graph_output
        ])

    # SNAP Coverage & Financial Indicators
    elif triggered_id == "tab-5-link":
        cleaned_data = data.dropna(subset=["region", "state", "avg_net_worth", "avg_black"])
        melted_data = cleaned_data.melt(
            id_vars=["region", "state"], 
            value_vars=["avg_net_worth", "avg_black"], 
            var_name="Metric", 
            value_name="value"
        )
        fig = px.treemap(
            melted_data,
            path=["region", "Metric", "state"],
            values="value",
            title="SNAP Coverage & Financial Indicators",
            color="value",
            color_continuous_scale="Inferno"  # Changed to use yellow, red, orange, etc.
        )
        return dcc.Graph(figure=fig)

    # Savings & Food Insecurity
    elif triggered_id == "tab-6-link":
        filtered_data = data.dropna(subset=["avg_emergency_savings", "percent_gap_cost_snap"])
        fig_area = px.area(
            filtered_data, x="state", y="percent_gap_cost_snap", title="Impact of Savings on SNAP Gaps"
        )
        savings_bins = pd.cut(
            filtered_data["avg_emergency_savings"], bins=4, labels=["Low", "Medium", "High", "Very High"]
        )
        fig_pie = px.pie(
            filtered_data, names=savings_bins, values="percent_gap_cost_snap", title="Food Insecurity by Savings Levels"
        )
        # Add toggle button for showing/hiding the pie chart
        return html.Div([
            dbc.Button("Show", id="toggle-pie-chart", color="primary", className="mb-3"),
            dbc.Row([
                dbc.Col(dcc.Graph(figure=fig_area, id="area-chart"), id="area-col", md=12),
                dbc.Col(dcc.Graph(figure=fig_pie, id="pie-chart", style={"display": "none"}), id="pie-col", md=4)
            ])
        ])

# Callback to handle the toggle button for Savings & Food Insecurity
@app.callback(
    [Output("pie-chart", "style"),
     Output("area-col", "md"),
     Output("toggle-pie-chart", "children")],
    Input("toggle-pie-chart", "n_clicks"),
    prevent_initial_call=True
)
def toggle_pie_chart(n_clicks):
    if n_clicks % 2 == 1:  # Odd clicks: Show the pie chart
        return {"display": "block"}, 8, "Hide"
    else:  # Even clicks: Hide the pie chart
        return {"display": "none"}, 12, "Show"


# Callback for Regional Insights Dropdown
@app.callback(
    Output("regional-insights-graph", "children"),
    Input("regional-chart-toggle", "value")
)
def update_regional_insights(selected_chart):
    if selected_chart == "heatmap":
        fig = px.imshow(
            data.pivot_table(
                values="normalized_cost_per_meal",
                index="region",
                columns="state",
                aggfunc="mean"
            ),
            aspect="auto",
            color_continuous_scale="Viridis",
            title="Heatmap of Food Insecurity by Region and State"
        )
    else:
        fig = px.box(
            data,
            x="region",
            y="normalized_cost_per_meal",
            color="region",
            title="Box Plot of Regional Variability in Food Insecurity"
        )
    return dcc.Graph(figure=fig)

# Callback for Homeownership Toggle Switch
@app.callback(
    Output("homeownership-graph", "children"),
    Input("chart-toggle", "value")
)
def update_homeownership_chart(toggle_value):
    race_columns = ["avg_aapi", "avg_black", "avg_hispanic", "avg_white"]
    grouped_data = data.groupby("state")[race_columns].mean().reset_index()

    if toggle_value:  # 3D Surface Chart
        z_values = np.array([grouped_data[race] for race in race_columns])
        fig = go.Figure(data=[go.Surface(z=z_values, x=grouped_data["state"], y=race_columns)])
        fig.update_layout(
            title="Homeownership Disparities (3D Surface Chart)",
            autosize=True,
            height=700,  # Increase the height for better visibility
            width=1000,  # Increase the width to match other graphs
            margin=dict(l=0, r=0, t=50, b=0),  # Remove unnecessary margins
            scene=dict(
                xaxis_title="State",
                yaxis_title="Race",
                zaxis_title="Homeownership Rate",
                xaxis=dict(showgrid=True, gridcolor="lightgray"),
                yaxis=dict(showgrid=True, gridcolor="lightgray"),
                zaxis=dict(showgrid=True, gridcolor="lightgray")
            )
        )
    else:  # Bar Chart
        melted_data = grouped_data.melt(id_vars="state", var_name="Race", value_name="Homeownership Rate")
        fig = px.bar(
            melted_data,
            x="state",
            y="Homeownership Rate",
            color="Race",
            title="Homeownership Disparities (Bar Chart)",
            barmode="group"
        )
    return dcc.Graph(figure=fig, style={"height": "700px", "width": "100%"})


# Run the app
if __name__ == "__main__":
    app.run_server(debug=True)
